In [1]:
import time, warnings
import time, re
warnings.filterwarnings('ignore')
from py_files.common_functions import *
from py_files.table_update import *
from py_files.get_information import *
from bs4 import BeautifulSoup as bs

%load_ext autoreload
%autoreload 2

########################################################################
## 발송계정 / 앱비밀번호 / 수신자는 notebooks/.env 에서 자동으로 읽어옵니다.
## 변경이 필요하면 notebooks/.env 를 수정한 뒤 이 셀을 다시 실행하세요.
## (비밀번호를 이 노트북에 직접 적지 마세요)
########################################################################
print(f"발송 계정   : {SEND_ADDR}")
print(f"수신자      : {', '.join(RECV_ADDRS)}")
print(f"다운로드 폴더: {DOWNLOAD_FOLDER}")

발송 계정   : runesys@naver.com
수신자      : h09144@koreainvestment.com, h09087@koreainvestment.com, hi1126@koreainvestment.com
다운로드 폴더: C:/Users/Administrator/Downloads


In [2]:
###########################################################
## 기존의 법령/규칙 데이터를 모두 업데이트 합니다.
## 이는 준법지원실의 법률리스트와 무관하게, 모든 법령과 규칙을 업데이트 합니다.
###########################################################
table_update()

law_2026-07-05 이후 구간을 수집합니다. (최근 60일)
    수집 400건 중 신규 0건
reg_2026-07-05 이후 구간을 수집합니다. (최근 60일)
    수집 1150건 중 신규 24건
데이터 업데이트 완료. 엑셀과 실제 웹페이지를 비교해 오류가 없는지 확인해주세요.


[                                             행정규칙명       행정규칙종류        발령번호  \
 0                                   유해간행물 심의 목록 고시    문화체육관광부고시   제2026-38호   
 1               2027 서울 세계청년대회 지원단의 설치 및 운영에 관한 규정       국무총리훈령       제951호   
 3                            해외수역 어획물 전재 허가에 관한 고시      해양수산부고시  제2026-104호   
 4              중증장애인생산품 구매목표 비율 및 생산시설 지정 등에 관한 기준      보건복지부고시  제2026-185호   
 6                         평택지방해양수산청 위임전결 및 사무관리 규정  평택지방해양수산청예규        제36호   
 12                           재정경제부 인공지능 및 데이터 관리규정      재정경제부훈령       제174호   
 15                자율기구 "규제서비스혁신추진단" 설치 및 운영에 관한 규정   식품의약품안전처훈령       제276호   
 24                                    병해충에 해당되는 잡초   농림축산검역본부고시   제2026-37호   
 28                               농식품인재개발원 정보공개운영지침   농식품인재개발원예규        제83호   
 29                         농식품인재개발원 인터넷 홈페이지 운영 지침   농식품인재개발원예규        제88호   
 30                                 농식품인재개발원 인사관리규정   농식품인재개발원예규        제86호   
 31                               농식품인재개

In [3]:
###########################################################
## 개정 후 안내하지 않은 법령을 리스트화하고, 법규리스트 개정일자를 최신화함
###########################################################
close_law_list_excel()   ## 열려있는 법규리스트 엑셀파일 저장안하고 강제 종료
update_df = get_update_df().sort_values('최근개정일', ascending=False)

print(f"개정 안내가 필요한 법률/규칙은 총 {len(update_df)}개 입니다. 아래 내용을 확인하세요.")
display(update_df)

개정 안내가 필요한 법률/규칙은 총 0개 입니다. 아래 내용을 확인하세요.


,site_category,law_name,최근발송일,최근개정일


In [ ]:
###########################################################
## 업데이트해야할 법령을 찾고 담당자에게 이메일 발송
###########################################################
browser = get_browser()

for idx, rows in update_df.iterrows():    
    site_category, law_name = rows['site_category'], rows['law_name']

    # if law_name != "자본시장과금융투자업에관한법률시행령":
    #     continue
        
    move_to_home(browser, site_category)
    time.sleep(0.5)
    name_col, date_col = get_column_name(site_category)    
    
    last_page_number = 200
    page_ranges = get_page_range(last_page_number)
   
    find_continue = True
    for page_range in page_ranges:  # 각 페이지 범위들을 돌면서 데이터를 크롤링함

        for page in range(page_range[0], page_range[1] + 1):
            while True:
                try:
                    move_to_page(browser, page)  ## 해당 페이지로 이동해서넵넵
                    break
                except:
                    time.sleep(1)
            time.sleep(0.5)

            while True:  ## 이전 테이블과 동일하면. 크롤링에 랙이 발생한것이므로 다시
                page_source = bs(browser.page_source, 'html.parser')
                table_df = get_page_table_info(page_source)  ## 테이블 데이터 긁고
                if page == 1:
                    break
                if table_df_prev[name_col].equals(table_df[name_col]):
                    continue
                else:
                    break
            ## 찾고자 하는 법령이 존재하면
            table_df[name_col] = table_df[name_col].map(lambda x: re.sub(r'[^가-힣]', '', x).strip())            
            if law_name in table_df[name_col].values.tolist():
                find_continue = False
                # 업데이트할 법령의 게시판 번호
                find_num = table_df[table_df[name_col]==law_name].번호.max() ## 제목이 같을 수 있어서. 무시하고 젤 최신으로
                update_date = table_df[table_df[name_col]==law_name][date_col].max()
                break            
            table_df_prev = table_df.copy()
        if find_continue == False:
            break
        if page_range[1] == last_page_number:
            break
        else:
            click_next_page(browser)
            time.sleep(0.5)
    ## 먼저 다운로드 폴더에 해당 법규명파일이 있으면 모두 삭제
    remove_dup_files()
    ## 이 시각 이후에 저장된 파일만 첨부 대상으로 삼음(다운로드 폴더의 개인 파일 오첨부 방지)
    download_started = time.time()
    ## 최종적으로 업데이트할 법령의 게시판 번호를 클릭하고
    click_law_row(browser, find_num=find_num, law_name=law_name)
    ## 제정/개정이유와 신구법비교 hwp 파일을 모두 가져옴
    mail_body = get_law_information(browser, law_name=law_name)
    ## 메일을 발송해야할 파일 리스트
    file_list = find_law_files(law_name, since=download_started)
    ## 메일 제목
    mail_title = f"({update_date}){law_name} 개정에 따른 안내"
    ## 메일 발송
    send_mail(mail_title, mail_body, file_list)

    ## 발송에 성공한 경우에만 최근발송일을 기록함.
    ## 이 기록이 없으면 다음 실행 때 같은 법령이 다시 대상으로 잡혀 중복 발송된다.
    mark_as_sent(law_name, update_date)
    print(f"발송 완료 : {mail_title}")

print("="*50)
print('모든 작업을 완료하였습니다.')
print("="*50)

In [4]:
################################################
## 과거 법령 찾을때만 돌리는 코드. 평소엔 할필요 없음
################################################
site_categories = ['law', 'reg']
for site_category in site_categories:
    get_total_table_info(site_category)